<a href="https://colab.research.google.com/github/Ivan137950/Samokat_kaggle/blob/main/samokat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Леонов Иван Алексеевич***

Решение от 22.05.2026

In [ ]:
# !pip install catboost
# !pip install optuna

import catboost as ctb
from catboost import CatBoostRegressor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Set, Iterable
# import optuna


# Preprocessing

## Метод пристального взгляда

*   `shifts_prediction.csv` - прогноз числа заказов на неделю и запланированная нагрузка (нормативное значение числа заказов на одного курьера в час), а также прогнозное число курьеров, чтобы обслужить эти заказы. Каждая дата - понедельник соответствующей недели, на которую приведены прогнозные значения.

*   `facts.csv` - файл содержит реальное кол-во курьеров, которое нужно было, чтобы обслужить все фактические заказы за неделю, предшествующую текущей (маркером текущей недели является также понедельник). Также содержит фактическую нагрузку за прошедшую неделю, число курьеров, предлагавших свои услуги (не обязательно вышедших на смены), фактическое кол-во заказов, лайфтайм даркстора, название города, общее число трудоустроенных курьеров, число курьеров, вышедших на смены, расходы на маркетинг, факт повышенного спроса.

*   `train.csv` - содержит для каждого даркстора на даты в прошлом дефицит курьеров в поле target. В файле, очевидно, не указаны значения для тестовой даты, на которую необходимо сделать прогноз!

*   `test.csv` - тестовый датасет, упорядочен по store_id. Необходимо сделать прогноз для каждого из приведенных в файле дарксторов (на единственную дату - неделю, следующую за последней известной в train.csv, т.е. на 24 ноября 2025 г.). Сама дата, на которую делается прогноз опущена в этом файле. Признаки для этой даты для каждого даркстора за соответствующую дату можно найти в приведенных выше файлах facts.csv, shifts_prediction.csv.

*   `sample_submission.csv` - пример файла для отправки решения.

In [2232]:
facts = pd.read_csv("/content/facts.csv")
# sample_submission = pd.read_csv("/content/sample_submission.csv")
shifts_prediction = pd.read_csv("/content/shifts_prediction.csv")
test = pd.read_csv("/content/test.csv")
train = pd.read_csv("/content/train.csv")

**shifts_prediction.csv:**

*   `calendar_dt` - дата в формате YYYY-MM-DD
*   `store_id` - идентификатор даркстора
*   `predicted_staff_value` - предсказанное число человек, необходимое, чтобы обслужить все заказы на неделе прогноза,
*   `predicted_num_orders` - предсказанное число заказов на неделю прогноза,
*   `predicted_load_factor` - предсказанная нагрузка на курьеров (число заказов в час, которое приходится на одного курьера) на неделю прогноза.

**facts.csv:**

*   `calendar_dt` - дата в формате YYYY-MM-DD
*   `store_id` - идентификатор даркстора
*   `fact_staff_value_lag_1` - фактическое число человек, необходимое, чтобы обслужить все заказы на прошлой неделе,
*   `fact_load_factor_lag_1` - фактическая нагрузка на курьеров на прошлой неделе,
*   `num_available_couriers_lag_1` - число курьеров, предложивших свои услуги за предыдущую неделю,
*   `fact_num_orders_lag_1` - фактическое число заказов за предыдущую неделю,
*   `fact_percent_lateness_lag_1` - процент опозданий за предыдущую неделю,
*   `store_lifetime_in_days` - число дней функционирования даркстора,
*   `city_nm` - город функционирования даркстора,
*   `fact_staff_churn` - число курьеров, уволившихся из компании,
*   `flag_high_load_lag_1` - признак повышенного спроса за предыдущую неделю,
*   `marketing_costs_lag_1` - затраты на маркетинг, осуществленные за предыдущую неделю,
*   `fact_couriers_with_shifts_lag_1` - число курьеров, выходивших на смены на предыдущей неделе.

In [2233]:
facts.head(3)

,calendar_dt,store_id,fact_staff_value_lag_1,fact_load_factor_lag_1,num_available_couriers_lag_1,fact_num_orders_lag_1,fact_percent_lateness_lag_1,city_nm,store_lifetime_in_days,fact_staff_churn,flag_high_load_lag_1,marketing_costs_lag_1,fact_couriers_with_shifts_lag_1
0,2025-11-03,000fade4-e8dc-11ed-b10a-08c0eb31fffb,1,0.526316,10,10,NaN,Ульяновск,888.0,1.0,1,3.683577e+08,19.0
1,2025-11-10,000fade4-e8dc-11ed-b10a-08c0eb31fffb,8,1.941176,13,33,69.565217,Ульяновск,895.0,1.0,1,NaN,17.0
2,2025-11-17,000fade4-e8dc-11ed-b10a-08c0eb31fffb,8,2.400000,12,36,66.666667,Ульяновск,902.0,4.0,1,4.254502e+10,15.0


In [2234]:
shifts_prediction.head(3)

,calendar_dt,store_id,predicted_staff_value,predicted_num_orders,predicted_load_factor
0,2024-01-01,000fade4-e8dc-11ed-b10a-08c0eb31fffb,12,270,2.85
1,2024-01-08,000fade4-e8dc-11ed-b10a-08c0eb31fffb,14,310,2.85
2,2024-01-15,000fade4-e8dc-11ed-b10a-08c0eb31fffb,15,370,2.96


In [2235]:
train.head(3)

,calendar_dt,store_id,target
0,2025-11-03,000fade4-e8dc-11ed-b10a-08c0eb31fffb,1.0
1,2025-11-10,000fade4-e8dc-11ed-b10a-08c0eb31fffb,1.0
2,2025-11-17,000fade4-e8dc-11ed-b10a-08c0eb31fffb,4.0


In [2236]:
test.head(3)

,store_id
0,000fade4-e8dc-11ed-b10a-08c0eb31fffb
1,0022f1b0-b8f8-11ee-b10b-08c0eb31fffb
2,00440ac1-6a1d-11eb-85a3-1c34dae33151


In [2237]:
print(f"facts.shape: {facts.shape}")
print(f"shifts_prediction.shape: {shifts_prediction.shape}")
print(f"test.shape: {test.shape}")
print(f"train.shape: {train.shape}")

facts.shape: (10660, 13)
shifts_prediction.shape: (223470, 5)
test.shape: (2438, 1)
train.shape: (8220, 3)


## *facts*: Борьба с пропусками

In [2238]:
'''
fact_load_factor_lag_1 - фактическая нагрузка на курьеров на прошлой неделе,
fact_percent_lateness_lag_1 - процент опозданий за предыдущую неделю,
marketing_costs_lag_1 - затраты на маркетинг, осуществленные за предыдущую неделю,
fact_couriers_with_shifts_lag_1 - число курьеров, выходивших на смены на предыдущей неделе.

Пропуски можно заменить средним про TRAIN
'''
facts.isnull().sum()

,0
calendar_dt,0
store_id,0
fact_staff_value_lag_1,0
fact_load_factor_lag_1,117
num_available_couriers_lag_1,0
fact_num_orders_lag_1,0
fact_percent_lateness_lag_1,2459
city_nm,0
store_lifetime_in_days,0
fact_staff_churn,0


In [2239]:
'''
marketing_costs_lag_1 -> log(marketing_costs_lag_1)
'''
facts.describe()

,fact_staff_value_lag_1,fact_load_factor_lag_1,num_available_couriers_lag_1,fact_num_orders_lag_1,fact_percent_lateness_lag_1,store_lifetime_in_days,fact_staff_churn,flag_high_load_lag_1,marketing_costs_lag_1,fact_couriers_with_shifts_lag_1
count,10660.000000,10543.000000,10660.000000,10660.000000,8201.000000,10660.000000,10660.000000,10660.000000,7.457000e+03,10543.000000
mean,7.206660,2.929684,10.861538,33.038743,81.179845,976.580769,1.853002,0.677580,4.140296e+10,12.689367
std,4.647654,1.969138,3.948154,18.533299,19.055068,584.030709,1.860373,0.467425,8.297097e+11,5.145462
min,1.000000,0.000000,0.000000,0.000000,2.702703,0.000000,0.000000,0.000000,7.559291e+03,1.000000
25%,4.000000,1.636364,9.000000,17.000000,71.428571,500.000000,0.000000,0.000000,6.126757e+07,9.000000
50%,7.000000,2.642857,10.000000,33.000000,85.714286,909.000000,1.000000,1.000000,4.499068e+08,12.000000
75%,10.000000,3.750000,12.000000,44.000000,100.000000,1450.000000,3.000000,1.000000,3.649713e+09,15.000000
max,40.000000,45.000000,44.000000,215.000000,100.000000,2527.000000,5.000000,1.000000,6.283868e+13,40.000000


In [2240]:
# Преобразование 'calendar_dt' в формат даты и создание
facts['calendar_dt'] = pd.to_datetime(facts['calendar_dt'])
train['calendar_dt'] = pd.to_datetime(train['calendar_dt'])
# facts['calendar_dt'] = pd.to_datetime(facts['calendar_dt'])

facts['week'] = facts['calendar_dt'].dt.isocalendar().week
facts['year'] = facts['calendar_dt'].dt.isocalendar().year
facts['total_week'] = facts['week'] + 52 * (facts['year'] - 2024)

TRAIN_BOARD = 80 # последняя неделя TRAIN
VAL_BOARD = 90 # последняя неделя VAL


train_df = facts[facts['total_week'] < TRAIN_BOARD]

# Колонки для заполнения пропусков
columns_to_impute = [
    'fact_percent_lateness_lag_1',
    'fact_load_factor_lag_1',
    'marketing_costs_lag_1',
    'fact_couriers_with_shifts_lag_1'
]

for col in columns_to_impute:
    city_week_mean = train_df.groupby(['city_nm', 'calendar_dt'])[col].transform('mean')
    facts[col] = facts[col].fillna(city_week_mean)
    facts[col] = facts[col].fillna(train_df[col].mean()) # Случай, когда для комбинации город-дата все значения NaN

print("Количество NaN в 'facts' после заполнения пропусков:")
print(facts[columns_to_impute].isnull().sum())

# Как и оговаривалось ранее, берем логарифм
facts['log_marketing_costs_lag_1'] =  np.log1p(facts['marketing_costs_lag_1'])

Количество NaN в 'facts' после заполнения пропусков:
fact_percent_lateness_lag_1        0
fact_load_factor_lag_1             0
marketing_costs_lag_1              0
fact_couriers_with_shifts_lag_1    0
dtype: int64


Добавляем таргеты

In [2241]:
train['calendar_dt'] = pd.to_datetime(train['calendar_dt'])
# JOIN features
df = pd.merge(facts, train, on=('calendar_dt', 'store_id'))
# + targets

# Первая модель. Обучение на данных facts


In [2242]:
train_data = df[df['total_week'] < TRAIN_BOARD]
val_data = df[
    (df['total_week'] >= TRAIN_BOARD) &
    (df['total_week'] < VAL_BOARD)
    ]
test_data = df[df['total_week'] >= VAL_BOARD]


In [2243]:
s = " ".join(df.columns.tolist())
print(f'[\n\t"{'",\n\t"'.join(s.split())}"]')


[
	"calendar_dt",
	"store_id",
	"fact_staff_value_lag_1",
	"fact_load_factor_lag_1",
	"num_available_couriers_lag_1",
	"fact_num_orders_lag_1",
	"fact_percent_lateness_lag_1",
	"city_nm",
	"store_lifetime_in_days",
	"fact_staff_churn",
	"flag_high_load_lag_1",
	"marketing_costs_lag_1",
	"fact_couriers_with_shifts_lag_1",
	"week",
	"year",
	"total_week",
	"log_marketing_costs_lag_1",
	"target"]


In [2244]:
columns = [
    # Хочу руками удалить лишние столбцы
	# "calendar_dt",
	"store_id",
	"fact_staff_value_lag_1",
	"fact_load_factor_lag_1",
	"num_available_couriers_lag_1",
	"fact_num_orders_lag_1",
	"fact_percent_lateness_lag_1",
	"city_nm",
	"store_lifetime_in_days",
	"fact_staff_churn",
	"flag_high_load_lag_1",
	"marketing_costs_lag_1",
	"fact_couriers_with_shifts_lag_1",
	"week",
	"year",
	"total_week",
	"log_marketing_costs_lag_1",
	# "target"
]

cat_features = [
    'city_nm',
    'store_id',
    'flag_high_load_lag_1'
    ]

X_train =  train_data[columns]
y_train = train_data['target']
X_val =  val_data[columns]
y_val = val_data['target']
X_test = test_data[columns]
y_test = test_data['target']


### Train


In [2245]:
def WAPE(y, y_pred):
    # Добавляю округление, так как, очевидно, дефицита в полтора сотрудника быть не может
    y_pred_rounded = np.round(y_pred)
    return np.sum(np.abs(y - y_pred_rounded)) / np.sum(y)

In [2246]:
model_catboost = CatBoostRegressor(
    learning_rate=0.005,
    depth=3,
    l2_leaf_reg=1,
    loss_function='MAE',
    eval_metric='MAE',
    verbose=False
    )

model_catboost.fit(
    X_train,
    y=y_train,
    cat_features=cat_features,
    use_best_model=True,
    eval_set=(X_val, y_val))


y_pred = model_catboost.predict(X_test)
wape = WAPE(y_test, y_pred)
print(f"WAPE на валидации: {wape:.4f}")

WAPE на валидации: 0.1793


In [2247]:
feature_scores = model_catboost.get_feature_importance()
df_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_scores
})
df_importance = df_importance.sort_values('Importance', ascending=False).reset_index(drop=True)
df_importance

,Feature,Importance
0,fact_staff_churn,56.299298
1,total_week,12.884817
2,flag_high_load_lag_1,9.274232
3,num_available_couriers_lag_1,8.758742
4,fact_staff_value_lag_1,3.665736
5,fact_num_orders_lag_1,2.764614
6,week,2.534258
7,year,0.921092
8,fact_load_factor_lag_1,0.848146
9,store_lifetime_in_days,0.499215


Уберу наименее значимые фичи

In [2248]:
most_important_features = df_importance[df_importance['Importance'] > 1]['Feature']

X_train_most_important =  train_data[most_important_features]
X_val_most_important =  val_data[most_important_features]
X_test_most_important = test_data[most_important_features]


In [2249]:
model_catboost_reduced = CatBoostRegressor(
    learning_rate=0.005,
    depth=3,
    l2_leaf_reg=1,
    loss_function='MAE',
    eval_metric='MAE',
    verbose=False
    )

model_catboost_reduced.fit(
    X_train_most_important,
    y=y_train,
    # cat_features=cat_features,
    use_best_model=True,
    eval_set=(X_val_most_important, y_val))


y_pred_most_important = model_catboost_reduced.predict(X_test_most_important)
wape = WAPE(y_test, y_pred_most_important)
print(f"WAPE на валидации: {wape:.4f}")

WAPE на валидации: 0.1744


# При использовании только *facts* **WAPE** на тестовых данных: `0.1744`. Замержим еще фич из *shifts_prediction* и посмотрим на результат

## Preprocessing

In [2250]:
'''
predicted_load_factor - предсказанная нагрузка на курьеров (число заказов в час, которое приходится на одного курьера) на неделю прогноза.
Рассмотрим, что происходит с остальными параметрами, когда predicted_load_factor is Nan
'''
shifts_prediction.isnull().sum()

,0
calendar_dt,0
store_id,0
predicted_staff_value,0
predicted_num_orders,0
predicted_load_factor,199


In [2251]:
'''
predicted_num_orders - предсказанное число заказов на неделю прогноза,
Меня гложат смутные сомнения: неужто predicted_num_orders == 0 при predicted_load_factor is NaN?
'''
shifts_prediction[shifts_prediction['predicted_load_factor'].isnull()]

,calendar_dt,store_id,predicted_staff_value,predicted_num_orders,predicted_load_factor
1175,2025-08-25,0181305a-1ff8-11ee-b971-08c0eb32008b,8,0,NaN
2145,2025-08-25,02853a18-3730-11ec-a0ee-ec0d9a21b021,8,0,NaN
3505,2025-08-25,03bcdafc-6d03-11ee-8860-08c0eb32014b,8,0,NaN
4121,2025-08-25,04793dd7-bd32-11ea-b7de-0050560306e1,5,0,NaN
4872,2025-08-25,057fbe65-f3d2-11ed-b971-08c0eb32008b,10,0,NaN
...,...,...,...,...,...
214568,2025-08-25,f48636dc-e543-11ef-a5c9-7cc255b74513,8,0,NaN
215717,2025-08-25,f6a07736-a1cf-11ef-ae7a-08c0eb320147,11,0,NaN
217507,2025-08-25,f8962948-205a-11ec-a0ee-ec0d9a21b021,9,0,NaN
219312,2025-08-25,fae955f9-8581-11eb-85a3-1c34dae33151,6,0,NaN


In [2252]:
'''
Как минимум ситуации, когда (predicted_num_orders > 0) & (predicted_load_factor is NaN) нет.
Утверждать что-либо однозначно нельзя. Но логически предположить, что отсутствие заказов означает
отсутствие нагрузки на курьеров будто бы можно. Ссылаясь на данное предположение, заменю пропуски нулями.
'''
len(shifts_prediction[(shifts_prediction['predicted_load_factor'].isnull()) & (shifts_prediction['predicted_num_orders'] > 0)])

0

In [2253]:
shifts_prediction['calendar_dt'] = pd.to_datetime(shifts_prediction['calendar_dt'])
shifts_prediction = shifts_prediction.fillna(0)

**JOIN** *shifts_prediction* to *df*



In [2254]:
full_df = df.merge(shifts_prediction, on=('calendar_dt', 'store_id'))

# Вторая модель

In [2255]:
full_train_data = full_df[full_df['total_week'] < TRAIN_BOARD]
full_val_data = full_df[
    (full_df['total_week'] >= TRAIN_BOARD) &
    (full_df['total_week'] < VAL_BOARD)
    ]
full_test_data = full_df[full_df['total_week'] >= VAL_BOARD]

In [2256]:
s = " ".join(full_df.columns.tolist())
print(f'[\n\t"{'",\n\t"'.join(s.split())}"]')


[
	"calendar_dt",
	"store_id",
	"fact_staff_value_lag_1",
	"fact_load_factor_lag_1",
	"num_available_couriers_lag_1",
	"fact_num_orders_lag_1",
	"fact_percent_lateness_lag_1",
	"city_nm",
	"store_lifetime_in_days",
	"fact_staff_churn",
	"flag_high_load_lag_1",
	"marketing_costs_lag_1",
	"fact_couriers_with_shifts_lag_1",
	"week",
	"year",
	"total_week",
	"log_marketing_costs_lag_1",
	"target",
	"predicted_staff_value",
	"predicted_num_orders",
	"predicted_load_factor"]


In [2257]:
columns_full = [
    # Хочу руками удалить лишние столбцы
    # "calendar_dt",
	"store_id",
	"fact_staff_value_lag_1",
	"fact_load_factor_lag_1",
	"num_available_couriers_lag_1",
	"fact_num_orders_lag_1",
	"fact_percent_lateness_lag_1",
	"city_nm",
	"store_lifetime_in_days",
	"fact_staff_churn",
	"flag_high_load_lag_1",
	"marketing_costs_lag_1",
	"fact_couriers_with_shifts_lag_1",
	"week",
	"year",
	"total_week",
	"log_marketing_costs_lag_1",
	# "target",
	"predicted_staff_value",
	"predicted_num_orders",
	"predicted_load_factor"
]

X_train_full = full_train_data[columns_full]
X_val_full   = full_val_data[columns_full]
X_test_full  = full_test_data[columns_full]


## train

In [2258]:
model_catboost_full = CatBoostRegressor(
    learning_rate=0.01,
    depth=3,
    l2_leaf_reg=5,
    loss_function='MAE',
    eval_metric='MAE',
    verbose=False
    )

# Исправляем: берем y из того же датафрейма, что и X
model_catboost_full.fit(
    X_train_full,
    y=full_train_data['target'],
    cat_features=cat_features,
    use_best_model=True,
    eval_set=(X_val_full, full_val_data['target']))


y_pred_full = model_catboost_full.predict(X_test_full)
wape = WAPE(full_test_data['target'], y_pred_full)
print(f"WAPE на валидации (full model): {wape:.4f}")

WAPE на валидации (full model): 0.1757


In [2259]:
feature_scores = model_catboost_full.get_feature_importance()
df_importance = pd.DataFrame({
    'Feature': X_train_full.columns,
    'Importance': feature_scores
})
df_importance = df_importance.sort_values('Importance', ascending=False).reset_index(drop=True)
df_importance

,Feature,Importance
0,fact_staff_churn,46.684931
1,num_available_couriers_lag_1,11.443221
2,total_week,11.281892
3,flag_high_load_lag_1,9.020603
4,predicted_staff_value,4.079856
5,fact_staff_value_lag_1,3.408930
6,fact_num_orders_lag_1,3.329503
7,predicted_load_factor,3.108357
8,week,2.530474
9,predicted_num_orders,1.286062


чистка от малозначимых фич

In [2260]:
most_important_features = df_importance[df_importance['Importance'] > 1.5]['Feature']
#Лучший результат достигается при df_importance['Importance'] > 1.5
X_train_most_important_full = full_train_data[most_important_features]
X_val_most_important_full = full_val_data[most_important_features]
X_test_most_important_full = full_test_data[most_important_features]

model_catboost_reduced_full = CatBoostRegressor(
    learning_rate=0.05,
    depth=3,
    l2_leaf_reg=0.84,
    loss_function='MAE',
    eval_metric='MAE',
    verbose=False
    )

model_catboost_reduced_full.fit(
    X_train_most_important_full,
    y=full_train_data['target'],
    # cat_features=cat_features,
    use_best_model=True,
    eval_set=(X_val_most_important_full, full_val_data['target']))


y_pred_most_important_full = model_catboost_reduced_full.predict(X_test_most_important_full)
wape = WAPE(full_test_data['target'], y_pred_most_important_full)
print(f"WAPE на валидации (reduced full model): {wape:.4f}")

WAPE на валидации (reduced full model): 0.1715


По итогу получили точность в `0.1715` на тестовой выборке

Оценим, что осталось от изначального датасета

In [2261]:
full_df[most_important_features].head(3)

,fact_staff_churn,num_available_couriers_lag_1,total_week,flag_high_load_lag_1,predicted_staff_value,fact_staff_value_lag_1,fact_num_orders_lag_1,predicted_load_factor,week
0,1.0,10,97,1,7,1,10,4.96,45
1,1.0,13,98,1,6,8,33,5.06,46
2,4.0,12,99,1,7,8,36,4.86,47


# Предсказания для *test*

In [2262]:
# dataset for test data
facts_for_test = facts[facts['total_week'] == 100]
df_for_test = facts_for_test.merge(shifts_prediction, on=('calendar_dt', 'store_id'))
df_for_test = df_for_test.merge(test, on=('store_id'))

# чистим данные для test data
df_clean_for_test = df_for_test[most_important_features]

# prediction
df_for_test['target'] = np.round(model_catboost_reduced_full.predict(df_clean_for_test))
df_for_test.head(3)

,calendar_dt,store_id,fact_staff_value_lag_1,fact_load_factor_lag_1,num_available_couriers_lag_1,fact_num_orders_lag_1,fact_percent_lateness_lag_1,city_nm,store_lifetime_in_days,fact_staff_churn,...,marketing_costs_lag_1,fact_couriers_with_shifts_lag_1,week,year,total_week,log_marketing_costs_lag_1,predicted_staff_value,predicted_num_orders,predicted_load_factor,target
0,2025-11-24,000fade4-e8dc-11ed-b10a-08c0eb31fffb,11,2.294118,12,39,94.444444,Ульяновск,909.0,2.0,...,3.081126e+10,17.0,48,2025,100,24.151146,8,320,4.66,2.0
1,2025-11-24,0022f1b0-b8f8-11ee-b10b-08c0eb31fffb,6,2.727273,7,30,92.857143,Набережные Челны,638.0,0.0,...,2.001767e+05,11.0,48,2025,100,12.206961,5,230,6.47,0.0
2,2025-11-24,00440ac1-6a1d-11eb-85a3-1c34dae33151,10,3.000000,11,39,93.103448,Новосибирск,1700.0,0.0,...,7.958631e+06,13.0,48,2025,100,15.889768,9,340,4.47,1.0


In [2263]:
# Вспомним про формат вывода
pd.read_csv("/content/sample_submission.csv")


,store_id,target
0,000fade4-e8dc-11ed-b10a-08c0eb31fffb,0
1,0022f1b0-b8f8-11ee-b10b-08c0eb31fffb,0
2,00440ac1-6a1d-11eb-85a3-1c34dae33151,4
3,00442959-9671-11ec-ae6d-08c0eb320147,3
4,00562194-569a-11ec-a0ee-ec0d9a21b021,0
...,...,...
2433,ff4bc29d-715c-11ed-b96e-08c0eb32008b,0
2434,ff55434c-800d-11eb-85a3-1c34dae33151,0
2435,ff62435c-8278-11f0-9bb2-be3af2b6059f,0
2436,ffc39ff7-2cbe-11ec-a0ee-ec0d9a21b021,1


In [2266]:
df_for_test[['store_id', 'target']].to_csv("solution.csv")